# 05 - Risk Classification
**Vehicle Predictive Maintenance Project**

---

## Purpose of This Notebook

The goal of this notebook is to produce a single **High / Medium / Low risk score** for every active vehicle in the fleet. This gives the business a simple, actionable prioritisation — rather than trying to interpret multiple probabilities and flags, a fleet manager can immediately see which vehicles need attention first.

### Approach
We use **XGBoost**, a gradient boosting classifier, which is well suited to this problem because:
- It handles missing values natively (no tracker vehicles, no MOT dates)
- It works well with mixed feature types (numeric, encoded categorical)
- It produces feature importance scores so we can understand what's driving risk
- It is robust to skewed distributions like the ones we saw in feature engineering

### Labelling Strategy
We don't have a pre-existing 'risk' label in the data — we need to create one. We do this by combining signals from notebook 04:
- Survival model probabilities (Engine, Electrical, Cooling)
- Rule-based flags (Brakes, Wheels & Tyres)
- Supporting features (repair frequency, spend, days since last repair, mileage)

A composite risk score is calculated per vehicle and then bucketed into High / Medium / Low. The classifier then learns the pattern behind these labels so it can generalise to new vehicles.

### Important Note on Labelling
Because we are creating our own labels rather than using ground truth, this is a **semi-supervised** approach. The model is learning to replicate and generalise our scoring logic — not predicting a truly independent outcome. This is appropriate and honest given the data available, and is clearly documented here for transparency.

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

PROCESSED = '../data/processed/'
OUTPUTS   = '../outputs/'

features         = pd.read_csv(PROCESSED + 'features.csv')
features_active  = pd.read_csv(PROCESSED + 'features_active.csv')
survival_preds   = pd.read_csv(PROCESSED + 'survival_predictions.csv')

print('Data loaded.')
print(f'  All vehicles (features):    {len(features):,}')
print(f'  Active vehicles:            {len(features_active):,}')
print(f'  Survival predictions:       {len(survival_preds):,}')

## 2. Build the Composite Risk Score

### Why
Before we can train a classifier we need labels. We create a composite risk score by combining the signals from notebook 04 with supporting vehicle features. Each signal contributes points to a total score which is then bucketed into High / Medium / Low.

**Scoring logic:**
- **Electrical repair probability >15%** → +3 points (high probability, most common fault)
- **Electrical repair probability >8%** → +1 point
- **Engine or Cooling repair probability >5%** → +2 points each
- **Brakes flag** → +3 points (safety critical)
- **Tyres flag** → +3 points (safety critical)
- **Both brakes AND tyres flagged** → additional +2 points
- **Repairs in last 12 months > fleet median** → +1 point (already a busy repair vehicle)
- **Days since last repair > 365** → +1 point (overdue for any attention)
- **Vehicle age > 7 years** → +1 point (older vehicles generally higher risk)
- **Driver score below fleet average** → +1 point

**Bucketing:**
- Score ≥ 5 → High
- Score 2-4 → Medium  
- Score 0-1 → Low

In [ ]:
# Join survival predictions to active features
df = features_active.merge(survival_preds, on='Registration', how='left', suffixes=('', '_surv'))

print(f'Joined dataset shape: {df.shape}')
print(f'Columns from survival predictions joined: {[c for c in survival_preds.columns if c != "Registration"]}')

In [ ]:
# Fleet benchmarks for relative scoring
fleet_median_repairs_12m = df['Repairs_Last_12M_Log'].median()
fleet_avg_driver_score   = df['DriverScore'].mean()

print(f'Fleet median repairs last 12m (log): {fleet_median_repairs_12m:.2f}')
print(f'Fleet average driver score:          {fleet_avg_driver_score:.1f}')

In [ ]:
# Build composite risk score
score = pd.Series(0, index=df.index)

# Electrical probability
score += np.where(df['Prob_Electrical_90d'] > 0.15, 3,
         np.where(df['Prob_Electrical_90d'] > 0.08, 1, 0))

# Engine probability
score += np.where(df['Prob_Engine_90d'] > 0.05, 2, 0)

# Cooling probability
score += np.where(df['Prob_Cooling_90d'] > 0.05, 2, 0)

# Brakes flag (safety critical)
score += np.where(df['Brakes_Flag'] == 1, 3, 0)

# Tyres flag (safety critical)
score += np.where(df['Wheels_Tyres_Flag'] == 1, 3, 0)

# Both brakes AND tyres flagged — extra penalty
score += np.where((df['Brakes_Flag'] == 1) & (df['Wheels_Tyres_Flag'] == 1), 2, 0)

# Recent repair frequency above fleet median
score += np.where(df['Repairs_Last_12M_Log'] > fleet_median_repairs_12m, 1, 0)

# Long gap since any repair
score += np.where(df['Days_Since_Last_Repair'] > 365, 1, 0)

# Older vehicle
score += np.where(df['Vehicle Age (Years)'] > 7, 1, 0)

# Below average driver score
score += np.where(df['DriverScore'] < fleet_avg_driver_score, 1, 0)

df['Risk_Score'] = score

print('Risk score distribution:')
print(df['Risk_Score'].value_counts().sort_index())
print(f'\nMin: {df["Risk_Score"].min()}, Max: {df["Risk_Score"].max()}, Mean: {df["Risk_Score"].mean():.1f}')

In [ ]:
# Bucket into High / Medium / Low
def assign_risk_label(score):
    if score >= 5:
        return 'High'
    elif score >= 2:
        return 'Medium'
    else:
        return 'Low'

df['Risk_Label'] = df['Risk_Score'].apply(assign_risk_label)

print('Risk label distribution:')
print(df['Risk_Label'].value_counts())
print()
print('As percentage of active fleet:')
print((df['Risk_Label'].value_counts() / len(df) * 100).round(1).astype(str) + '%')

In [ ]:
# Visualise risk distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score histogram
df['Risk_Score'].plot(kind='hist', bins=range(0, int(df['Risk_Score'].max()) + 2),
                      ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(5, color='red', linestyle='--', label='High threshold (5)')
axes[0].axvline(2, color='orange', linestyle='--', label='Medium threshold (2)')
axes[0].set_title('Distribution of Composite Risk Scores')
axes[0].set_xlabel('Risk Score')
axes[0].legend()

# Label pie chart
label_counts = df['Risk_Label'].value_counts()
colors = {'High': '#e74c3c', 'Medium': '#f39c12', 'Low': '#2ecc71'}
pie_colors = [colors[l] for l in label_counts.index]
axes[1].pie(label_counts.values, labels=label_counts.index, colors=pie_colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Active Fleet Risk Distribution')

plt.tight_layout()
plt.savefig(OUTPUTS + 'reports/05_risk_distribution.png', dpi=150)
plt.show()

## 3. Prepare Features for the Classifier

### Why
We now train an XGBoost classifier using the composite risk labels we just created. The classifier learns the combination of features that leads to each risk level — this means it can be applied to future vehicles or updated data without manually recalculating the composite score each time.

We select a clean set of numeric features, excluding any columns that were directly used to create the labels (to avoid leakage) and any string columns the model can't use directly.

In [ ]:
# Define model features — exclude label-derived columns and non-numeric
EXCLUDE_COLS = [
    'Registration', 'Vehicle Status', 'Make', 'Asset Type', 'Driver',
    'Branch', 'Risk_Score', 'Risk_Label',
    # Exclude raw survival prob cols — too directly linked to label creation
    'Prob_Engine_90d', 'Prob_Electrical_90d', 'Prob_Cooling_90d',
    'Brakes_Flag', 'Wheels_Tyres_Flag',
    'Vehicle Status_surv'
]

feature_cols = [
    c for c in df.columns
    if c not in EXCLUDE_COLS
    and df[c].dtype in ['float64', 'int64', 'float32', 'int32']
]

print(f'Features for classifier: {len(feature_cols)}')
for col in feature_cols:
    print(f'  {col}')

In [ ]:
X = df[feature_cols].copy()
y = df['Risk_Label'].copy()

# Encode labels as integers for XGBoost
le = LabelEncoder()
y_encoded = le.fit_transform(y)
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

print(f'Label encoding: {label_mapping}')
print(f'\nClass distribution:')
for label, count in zip(*np.unique(y_encoded, return_counts=True)):
    print(f'  {le.inverse_transform([label])[0]:6s} ({label}): {count:>3} vehicles')

# Train / test split — stratified to preserve class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'\nTrain size: {len(X_train):,} | Test size: {len(X_test):,}')

## 4. Train XGBoost Classifier

### Why
XGBoost is a tree-based ensemble model that builds decision trees sequentially, with each tree correcting the errors of the previous one. Key advantages for our use case:
- Handles missing values without imputation
- Naturally captures non-linear relationships between features
- Provides feature importance which helps us validate the model makes sense
- Fast to train even with many features

We use cross-validation to get a reliable estimate of model performance before evaluating on the held-out test set.

In [ ]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')

print(f'Cross-validation accuracy (5-fold):')
print(f'  Per fold: {[round(s, 3) for s in cv_scores]}')
print(f'  Mean:     {cv_scores.mean():.3f}')
print(f'  Std:      {cv_scores.std():.3f}')

In [ ]:
# Train final model on full training set
model.fit(X_train, y_train)

# Evaluate on held-out test set
y_pred = model.predict(X_test)

print('Test set performance:')
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_
))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Risk Classification')
plt.tight_layout()
plt.savefig(OUTPUTS + 'reports/05_confusion_matrix.png', dpi=150)
plt.show()

## 5. Feature Importance

### Why
Feature importance tells us which inputs the model relied on most when making its risk predictions. This is crucial for two reasons:
1. **Validation** — the top features should make intuitive sense. If something unexpected is at the top it warrants investigation
2. **Future improvement** — low importance features could potentially be dropped in future versions to simplify the model

In [ ]:
importance_df = pd.DataFrame({
    'Feature':   feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 20
top_n = min(20, len(importance_df))
top_features = importance_df.head(top_n)

plt.figure(figsize=(12, 7))
sns.barplot(data=top_features, x='Importance', y='Feature', palette='muted')
plt.title(f'Top {top_n} Feature Importances — XGBoost Risk Classifier')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig(OUTPUTS + 'reports/05_feature_importance.png', dpi=150)
plt.show()

print('Top 10 most important features:')
print(importance_df.head(10).to_string(index=False))

## 6. Generate Risk Predictions for All Active Vehicles

### Why
Now we apply the trained model to all 715 active vehicles to generate their final risk labels. We also keep the composite risk score alongside the model prediction — the score gives a more granular view within each risk band which is useful for prioritisation in the output report.

In [ ]:
# Generate predictions for all active vehicles
X_all = df[feature_cols].copy()
df['Risk_Label_Predicted'] = le.inverse_transform(model.predict(X_all))

# Prediction probabilities — useful for ranking within risk bands
proba = model.predict_proba(X_all)
for i, cls in enumerate(le.classes_):
    df[f'Prob_{cls}'] = proba[:, i]

print('Predicted risk distribution (all active vehicles):')
print(df['Risk_Label_Predicted'].value_counts())
print()

# Compare composite label vs model prediction
agreement = (df['Risk_Label'] == df['Risk_Label_Predicted']).mean()
print(f'Agreement between composite score and model prediction: {agreement:.1%}')
print('>> High agreement confirms the model has learned the scoring logic well.')
print('>> Some disagreement is expected and healthy — the model generalises beyond the exact thresholds.')

In [ ]:
# Final risk output table
risk_output = df[[
    'Registration', 'Risk_Score', 'Risk_Label', 'Risk_Label_Predicted',
    'Prob_High', 'Prob_Medium', 'Prob_Low',
    'Vehicle Age (Years)', 'DriverScore',
    'Repairs_Last_12M_Log', 'Days_Since_Last_Repair',
    'Brakes_Flag', 'Wheels_Tyres_Flag'
]].copy()

# Sort by risk — High first, then by score descending within each band
risk_order = {'High': 0, 'Medium': 1, 'Low': 2}
risk_output['_sort'] = risk_output['Risk_Label_Predicted'].map(risk_order)
risk_output = risk_output.sort_values(['_sort', 'Risk_Score'], ascending=[True, False])
risk_output = risk_output.drop(columns=['_sort'])

print(f'Risk output table: {risk_output.shape}')
print('\nTop 10 highest risk vehicles:')
display(risk_output.head(10))

## 7. Risk Profile Analysis

### Why
Before saving we do a quick sense check — do High risk vehicles genuinely look different from Low risk ones across the key features? If the separation is clear it validates our approach.

In [ ]:
# Average feature values by risk band
profile_cols = [
    'Vehicle Age (Years)', 'DriverScore', 'Days_Since_Last_Repair',
    'Repairs_Last_12M_Log', 'Avg_Daily_Miles_Recent',
    'Brakes_Flag', 'Wheels_Tyres_Flag'
]

profile = df.groupby('Risk_Label_Predicted')[profile_cols].mean().round(2)
profile = profile.reindex(['High', 'Medium', 'Low'])

print('Average feature values by risk band:')
display(profile)

In [ ]:
# Box plots — key features by risk band
plot_features = ['Vehicle Age (Years)', 'Days_Since_Last_Repair',
                 'DriverScore', 'Avg_Daily_Miles_Recent']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
risk_palette = {'High': '#e74c3c', 'Medium': '#f39c12', 'Low': '#2ecc71'}

for ax, col in zip(axes, plot_features):
    sns.boxplot(
        data=df, x='Risk_Label_Predicted', y=col,
        order=['High', 'Medium', 'Low'],
        palette=risk_palette, ax=ax
    )
    ax.set_title(col)
    ax.set_xlabel('Risk Band')

plt.suptitle('Feature Distributions by Risk Band', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS + 'reports/05_risk_profiles.png', dpi=150)
plt.show()

## 8. Save Outputs

In [ ]:
# Save risk output table
risk_output.to_csv(PROCESSED + 'risk_scores.csv', index=False)

# Save trained model
joblib.dump(model, OUTPUTS + 'models/xgb_risk_classifier.pkl')
joblib.dump(le, OUTPUTS + 'models/risk_label_encoder.pkl')

print('Outputs saved:')
print(f'  risk_scores.csv              — {len(risk_output):,} active vehicles')
print(f'  xgb_risk_classifier.pkl      — trained XGBoost model')
print(f'  risk_label_encoder.pkl       — label encoder')
print()
print('Final risk summary:')
for label in ['High', 'Medium', 'Low']:
    count = (risk_output['Risk_Label_Predicted'] == label).sum()
    pct   = count / len(risk_output) * 100
    print(f'  {label:6s}: {count:>3} vehicles ({pct:.1f}%)')

## 9. Summary

In [ ]:
print("""
========================================================
 RISK CLASSIFICATION SUMMARY
========================================================

LABELLING APPROACH:
  Composite risk score from:
  - Electrical/Engine/Cooling 90-day repair probabilities
  - Brakes and Tyres rule-based flags (safety critical)
  - Recent repair frequency vs fleet median
  - Days since last repair
  - Vehicle age
  - Driver score vs fleet average

  High:   score >= 5
  Medium: score 2-4
  Low:    score 0-1

MODEL:
  - XGBoost classifier
  - 5-fold cross-validation on training set
  - Evaluated on 20% held-out test set

IMPORTANT NOTE:
  Labels are derived from our own scoring logic, not ground
  truth outcomes. The model learns to generalise this logic.
  This is appropriate given available data and is documented
  here for full transparency.

OUTPUTS:
  - risk_scores.csv
  - xgb_risk_classifier.pkl
  - risk_label_encoder.pkl
  - 05_risk_distribution.png
  - 05_confusion_matrix.png
  - 05_feature_importance.png
  - 05_risk_profiles.png

NEXT: Notebook 06 — Maintenance Schedule Output
========================================================
""")